In [260]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [261]:
df = pd.read_csv("../datasets/processed_dataset_train.csv")

In [262]:
df

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,ram,talk_time,three_g,touch_screen,wifi,price_range,PixelArea,ScreenArea
0,842,0,2.2,0,1,0,7,0.6,188,2,2,2549,19,0,0,1,1,15.120,63
1,1021,1,0.5,1,0,1,53,0.7,136,3,6,2631,7,1,1,0,2,1799.140,51
2,563,1,0.5,1,2,1,41,0.9,145,5,6,2603,9,1,1,0,2,2167.308,22
3,615,1,2.5,0,0,0,10,0.8,131,6,9,2769,11,1,0,0,2,2171.776,128
4,1821,1,1.2,0,13,1,44,0.6,141,2,14,1411,15,1,1,0,1,1464.096,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,14,668,19,1,1,0,0,2309.580,52
1996,1965,1,2.6,1,0,0,39,0.2,187,4,3,2032,16,1,1,1,2,1797.975,110
1997,1911,0,0.9,1,1,1,36,0.7,108,8,3,3057,5,1,1,0,3,1416.576,9
1998,1512,0,0.9,0,4,1,46,0.1,145,5,5,869,19,1,1,1,0,225.120,180


In [263]:
from sklearn.model_selection import train_test_split

x = df.drop(["price_range"], axis = 1)
y = df["price_range"]

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42)

In [264]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import FunctionTransformer

In [265]:
transformer = ColumnTransformer([
    ("trf1", StandardScaler(), ["battery_power", "clock_speed", "fc", "int_memory", "m_dep", "mobile_wt", "pc", "ram", "talk_time", "PixelArea", "ScreenArea"])
], remainder="passthrough")

In [266]:
x_train = transformer.fit_transform(x_train)
x_val = transformer.transform(x_val)

In [267]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

def metrics(y_test, y_pred):
    print("accuracy_score : ", accuracy_score(y_test, y_pred))
    print("precision_score : ", precision_score(y_test, y_pred, average="macro"))
    print("Recall_score : ", recall_score(y_test, y_pred,  average="macro"))
    print("F1 Score : ", f1_score(y_test, y_pred,  average="macro"))

In [268]:
def model_training(x_train, x_test, y_train,y_test,model):
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    metrics(y_test, pred)
    return pred

In [269]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(max_depth=10)
y_pred = model_training(x_train, x_val, y_train, y_val, model_dt)

accuracy_score :  0.8083333333333333
precision_score :  0.8102081122541815
Recall_score :  0.8074956750097146
F1 Score :  0.8084354911848138


In [270]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

model_svc = CalibratedClassifierCV(SVC(max_iter=700))
y_pred= model_training(x_train, x_val, y_train, y_val, model_svc)

accuracy_score :  0.8883333333333333
precision_score :  0.8885832657745396
Recall_score :  0.8876796849276314
F1 Score :  0.8879525920766331


In [271]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier()
y_pred= model_training(x_train, x_val, y_train, y_val, model_xgb)

accuracy_score :  0.8966666666666666
precision_score :  0.8967682735904974
Recall_score :  0.8961973482409491
F1 Score :  0.896110216442199


In [272]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(max_depth=15, n_jobs = -1)
y_pred= model_training(x_train, x_val, y_train, y_val, model_rf)

accuracy_score :  0.8766666666666667
precision_score :  0.8761942524481059
Recall_score :  0.8761292430402912
F1 Score :  0.8758714748035137


In [273]:
from sklearn.naive_bayes import GaussianNB

model_nb = GaussianNB()
y_pred= model_training(x_train, x_val, y_train, y_val, model_nb)

accuracy_score :  0.7783333333333333
precision_score :  0.7790179742343594
Recall_score :  0.777040197950216
F1 Score :  0.7778239953756364


In [274]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors=7)
y_pred= model_training(x_train, x_val, y_train, y_val, model_knn)

accuracy_score :  0.5783333333333334
precision_score :  0.6043037953957494
Recall_score :  0.5772609092852616
F1 Score :  0.5838951677326665


In [275]:
from sklearn.neural_network import MLPClassifier

model_mlp = MLPClassifier(hidden_layer_sizes=(128, ), activation="relu", solver = "adam", max_iter=1000)
y_pred= model_training(x_train, x_val, y_train, y_val, model_mlp)

accuracy_score :  0.8783333333333333
precision_score :  0.8782697893854927
Recall_score :  0.8773723915303984
F1 Score :  0.8769921991677211


In [276]:
# the final models which are performing better in the training and validation dataset are - MLPClassifier, rf. xgb, svc
# the plan is to make a soft boundary voting ensemble model with these 4 and find the performance of these 5 with respect to the test dataset 

In [277]:
from sklearn.ensemble import VotingClassifier

model_voting = VotingClassifier(
    estimators=[("mlp", model_mlp), ("rf", model_rf),("xgb", model_xgb), ("svc", model_svc)],
    voting = "soft"
)

y_pred= model_training(x_train, x_val, y_train, y_val, model_voting)

accuracy_score :  0.9133333333333333
precision_score :  0.9142699518404162
Recall_score :  0.912810789099846
F1 Score :  0.9127511556674077


In [278]:
df_test = pd.read_csv("../datasets/processed_dataset_test.csv")

In [279]:
df_test

,id,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,ram,talk_time,three_g,touch_screen,wifi,PixelArea,ScreenArea
0,1,1043,1,1.8,1,14,0,5,0.1,193,3,16,3476,2,0,1,0,319.112,84
1,2,841,1,0.5,1,4,1,61,0.8,191,5,12,3895,7,1,0,0,639.322,0
2,3,1807,1,2.8,0,1,0,27,0.9,186,3,4,2396,10,0,1,1,1734.820,170
3,4,1546,0,0.5,1,18,1,25,0.5,96,8,20,3893,7,1,1,0,516.840,0
4,5,1434,0,1.4,0,11,1,49,0.5,108,6,18,1773,7,1,0,1,606.690,120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,1700,1,1.9,0,0,1,54,0.5,170,7,17,2121,15,1,1,0,587.972,112
996,997,609,0,1.8,1,0,0,13,0.9,186,4,2,1933,19,0,1,1,1880.064,8
997,998,1185,0,1.4,0,1,1,8,0.5,80,1,12,1223,14,1,0,0,393.525,0
998,999,1533,1,0.5,1,0,0,50,0.4,171,2,12,2509,6,0,1,0,31.616,165


In [280]:
y_test = df["price_range"]
x_test = df.drop(["price_range"], axis = 1)


In [282]:
x_test = transformer.transform(x_test)

In [283]:
pred = model_voting.predict(x_test)
metrics(y_test, pred)

accuracy_score :  0.974
precision_score :  0.9741708972071159
Recall_score :  0.974
F1 Score :  0.9740099481313307


In [285]:
import pickle 

with open("model.pkl", "wb") as f:
    pickle.dump(model_voting, f)
